In [1]:
import pandas as pd

df = pd.read_csv("wildfires_small.csv")

df.head()

,FIRE_YEAR,DISCOVERY_DOY,DISCOVERY_TIME,LATITUDE,LONGITUDE,STATE,NWCG_CAUSE_CLASSIFICATION,NWCG_GENERAL_CAUSE,OWNER_DESCR,FIRE_SIZE,FIRE_SIZE_CLASS,DISCOVERY_DATE,CONT_DATE
0,1999,106,NaN,44.100000,-69.366667,ME,Missing data/not specified/undetermined,Missing data/not specified/undetermined,STATE,0.20,A,4/16/1999,4/16/1999
1,1996,198,1130.0,48.267500,-105.708300,MT,Human,Equipment and vehicle use,BIA,169.50,D,7/16/1996,7/16/1996
2,1996,256,1355.0,45.733333,-115.615000,ID,Natural,Natural,USFS,0.10,A,9/12/1996,9/12/1996
3,2008,357,NaN,21.396166,-157.984344,HI,Missing data/not specified/undetermined,Missing data/not specified/undetermined,MISSING/NOT SPECIFIED,0.25,A,12/22/2008,NaN
4,2005,328,NaN,33.241062,-84.048659,GA,Human,Missing data/not specified/undetermined,OTHER FEDERAL,47.87,C,11/24/2005,NaN


In [2]:
# Basic dataset information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (100000, 13)

Columns:
['FIRE_YEAR', 'DISCOVERY_DOY', 'DISCOVERY_TIME', 'LATITUDE', 'LONGITUDE', 'STATE', 'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'OWNER_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'DISCOVERY_DATE', 'CONT_DATE']

Missing values:
CONT_DATE                    38618
DISCOVERY_TIME               34115
FIRE_YEAR                        0
LATITUDE                         0
DISCOVERY_DOY                    0
LONGITUDE                        0
STATE                            0
NWCG_GENERAL_CAUSE               0
NWCG_CAUSE_CLASSIFICATION        0
OWNER_DESCR                      0
FIRE_SIZE                        0
FIRE_SIZE_CLASS                  0
DISCOVERY_DATE                   0
dtype: int64

Duplicate rows: 13


In [3]:
# Examine wildfire sizes
print(df["FIRE_SIZE"].describe())

print("\nFire size classes:")
print(df["FIRE_SIZE_CLASS"].value_counts().sort_index())

print("\nInvalid fire sizes:")
print((df["FIRE_SIZE"] <= 0).sum())

count    100000.000000
mean         87.769252
std        2902.091510
min           0.000100
25%           0.100000
50%           0.810000
75%           3.000000
max      537627.000000
Name: FIRE_SIZE, dtype: float64

Fire size classes:
FIRE_SIZE_CLASS
A    37866
B    48059
C    11198
D     1441
E      758
F      454
G      224
Name: count, dtype: int64

Invalid fire sizes:
0


In [4]:
#categorial size function 
def size_category(size):
    if size < 10:
        return "Small"
    elif size < 100:
        return "Medium"
    else:
        return "Large"

df["SIZE_CATEGORY"] = df["FIRE_SIZE"].apply(size_category)

print(df["SIZE_CATEGORY"].value_counts())

print("\nPercentages:")
print(df["SIZE_CATEGORY"].value_counts(normalize=True) * 100)

SIZE_CATEGORY
Small     85925
Medium    11198
Large      2877
Name: count, dtype: int64

Percentages:
SIZE_CATEGORY
Small     85.925
Medium    11.198
Large      2.877
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

feature = [
    "FIRE_YEAR",
    "DISCOVERY_DOY",
    "LATITUDE",
    "LONGITUDE",
    "STATE",
    "NWCG_CAUSE_CLASSIFICATION",
    "NWCG_GENERAL_CAUSE"
]

X = df[feature]
y = df["SIZE_CATEGORY"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining Categories:")
print(y_train.value_counts())

Training set: (80000, 7)
Testing set: (20000, 7)

Training Categories:
SIZE_CATEGORY
Small     68740
Medium     8958
Large      2302
Name: count, dtype: int64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "FIRE_YEAR",
    "DISCOVERY_DOY",
    "LATITUDE",
    "LONGITUDE"
]

categorical_features = [
    "STATE",
    "NWCG_CAUSE_CLASSIFICATION",
    "NWCG_GENERAL_CAUSE"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown = "ignore"), categorical_features)
])

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight={"Small": 1, "Medium": 3, "Large": 10},
        C=0.01,
        random_state=42
    ))
])
log_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['Large','Medium','Small']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['FIRE_YEAR','DISCOVERY_DOY','LATITUDE',...,'STATE', 'NWCG_CAUSE_CLASSIFICATION','NWCG_GENERAL_CAUSE']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = log_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, y_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.79775

Classification Report
              precision    recall  f1-score   support

       Large       0.10      0.18      0.13       575
      Medium       0.31      0.21      0.25      2240
       Small       0.88      0.89      0.89     17185

    accuracy                           0.80     20000
   macro avg       0.43      0.43      0.42     20000
weighted avg       0.80      0.80      0.80     20000


CConfusion Matrix
[[15376   973   836]
 [ 1631   473   136]
 [  406    63   106]]


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=2,
        class_weight={"Small": 1, "Medium": 5, "Large": 15},
        random_state=42,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['Large','Medium','Small']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['FIRE_YEAR','DISCOVERY_DOY','LATITUDE',...,'STATE', 'NWCG_CAUSE_CLASSIFICATION','NWCG_GENERAL_CAUSE']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'

In [10]:
rf_pred = rf_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, rf_pred))
print("\nClassification Report")
print(classification_report(y_test, rf_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, rf_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.77965

Classification Report
              precision    recall  f1-score   support

       Large       0.19      0.24      0.22       575
      Medium       0.28      0.43      0.34      2240
       Small       0.91      0.84      0.88     17185

    accuracy                           0.78     20000
   macro avg       0.46      0.50      0.48     20000
weighted avg       0.82      0.78      0.80     20000


CConfusion Matrix
[[14498  2254   433]
 [ 1132   955   153]
 [  282   153   140]]


In [11]:
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(
        class_weight = "balanced",
        max_iter = 1000,
        random_state = 42
    ))
])
svm_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['Large','Medium','Small']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['FIRE_YEAR','DISCOVERY_DOY','LATITUDE',...,'STATE', 'NWCG_CAUSE_CLASSIFICATION','NWCG_GENERAL_CAUSE']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'

In [12]:
svm_pred = svm_model.predict(X_test)

print("Accuracy", accuracy_score(y_test, svm_pred))
print("\nClassification Report")
print(classification_report(y_test, svm_pred))
print("\nCConfusion Matrix")
print(confusion_matrix(y_test, svm_pred, labels = ["Small", "Medium", "Large"]))

Accuracy 0.82345

Classification Report
              precision    recall  f1-score   support

       Large       0.13      0.08      0.10       575
      Medium       0.31      0.23      0.26      2240
       Small       0.88      0.93      0.90     17185

    accuracy                           0.82     20000
   macro avg       0.44      0.41      0.42     20000
weighted avg       0.80      0.82      0.81     20000


CConfusion Matrix
[[15916  1044   225]
 [ 1663   508    69]
 [  460    70    45]]


In [13]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size = 0.2,
    random_state = 42,
    stratify = y_train
)

print("Training set:", X_train_final.shape)
print("Validation:", X_val.shape)

Training set: (64000, 7)
Validation: (16000, 7)


In [14]:
from sklearn.metrics import f1_score
from sklearn.base import clone

for depth in [10, 15, 20, None]:
    model = clone(rf_model)
    model.set_params(classifier__max_depth = depth)

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average = "macro")

    print("Max Depth", depth, "F1", round(score, 4))

Max Depth 10 F1 0.4581
Max Depth 15 F1 0.4813
Max Depth 20 F1 0.4829
Max Depth None F1 0.4719


In [15]:
for leaf in [1, 2, 5, 10]:
    model = clone(rf_model)

    model.set_params(
        classifier__max_depth = 20,
        classifier__min_samples_leaf = leaf
    )

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average="macro")

    print("Min Samples Leaf:", leaf, "Macro F1:", round(score, 4))

Min Samples Leaf: 1 Macro F1: 0.4733
Min Samples Leaf: 2 Macro F1: 0.4829
Min Samples Leaf: 5 Macro F1: 0.4776
Min Samples Leaf: 10 Macro F1: 0.4757


In [16]:
for trees in [50, 100, 200, 300]:
    model = clone(rf_model)

    model.set_params(
        classifier__max_depth = 20,
        classifier__min_samples_leaf = 2,
        classifier__n_estimators = trees
    )

    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    score = f1_score(y_val, predictions, average="macro")

    print("Trees:", trees, "Macro F1:", round(score, 4))

Trees: 50 Macro F1: 0.4786
Trees: 100 Macro F1: 0.4773
Trees: 200 Macro F1: 0.4829
Trees: 300 Macro F1: 0.4799


In [17]:

best_rf = clone(rf_model)

best_rf.fit(X_train_final, y_train_final)

best_pred = best_rf.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, best_pred))

print("\nClassification Report:")
print(classification_report(y_val, best_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val,
    best_pred,
    labels=["Small", "Medium", "Large"]
))

Validation Accuracy: 0.785875

Classification Report:
              precision    recall  f1-score   support

       Large       0.20      0.25      0.22       460
      Medium       0.29      0.42      0.34      1792
       Small       0.91      0.85      0.88     13748

    accuracy                           0.79     16000
   macro avg       0.47      0.51      0.48     16000
weighted avg       0.82      0.79      0.80     16000


Confusion Matrix:
[[11708  1697   343]
 [  934   751   107]
 [  225   120   115]]


In [18]:
models = {
    "Logistic Regression": clone(log_model),
    "Random Forest": clone(best_rf),
    "Linear SVM": clone(svm_model)
}

for name, model in models.items():
    model.fit(X_train_final, y_train_final)
    predictions = model.predict(X_val)

    accuracy = accuracy_score(y_val, predictions)
    f1 = f1_score(y_val, predictions, average="macro")

    print(name)
    print("Accuracy:", round(accuracy, 4))
    print("Macro F1:", round(f1, 4))
    print()

Logistic Regression
Accuracy: 0.8113
Macro F1: 0.4337

Random Forest
Accuracy: 0.7859
Macro F1: 0.4829

Linear SVM
Accuracy: 0.8254
Macro F1: 0.428

